In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import json
import numpy as np
import timm
# Function to load idx2word and convert it to word2idx
def load_vocabulary(path):
    with open(path, 'r') as file:
        idx2word = json.load(file)
    word2idx = {v: int(k) for k, v in idx2word.items()}
    return idx2word, word2idx

# Load vocabulary
idx2word_path = '/home/vitoupro/code/image_captioning/notebook/idx2word.json'
idx2word, word2idx = load_vocabulary(idx2word_path)

# Attention Module
class Attention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super(Attention, self).__init__()
        self.attn = nn.Linear(encoder_dim + decoder_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1)

    def forward(self, encoder_out, hidden):
        hidden = hidden.unsqueeze(1).repeat(1, encoder_out.size(1), 1)
        attn_input = torch.cat((encoder_out, hidden), dim=2)
        energy = torch.tanh(self.attn(attn_input))
        attention = self.v(energy).squeeze(2)
        alpha = torch.softmax(attention, dim=1)
        context = (encoder_out * alpha.unsqueeze(2)).sum(dim=1)
        return context, alpha

class EncoderDeiT(nn.Module):
    def __init__(self, model_name='deit_small_patch16_224', pretrained=True):
        super(EncoderDeiT, self).__init__()
        self.model = timm.create_model(model_name, pretrained=pretrained, features_only=True)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((14, 14))  # Ensure consistent shape
        self.output_dim = self.model.feature_info[-1]['num_chs']  # e.g., 384 for deit_small

    def forward(self, images):
        # Extract feature maps
        features = self.model(images)[-1]  # (B, C, H, W), e.g., (B, 384, 14, 14)
        features = self.adaptive_pool(features)  # (B, C, 14, 14) ensures fixed size
        features = features.permute(0, 2, 3, 1)  # (B, 14, 14, C)
        features = features.view(features.size(0), -1, features.size(-1))  # (B, 196, C)
        return features
    
class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, attention_dim=256, encoder_dim=2048, num_layers=1, dropout_prob=0.3):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.attention = Attention(encoder_dim, hidden_size, attention_dim)
        self.lstm = nn.LSTMCell(embed_size + encoder_dim, hidden_size)
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(dropout_prob)
        self.init_h = nn.Linear(encoder_dim, hidden_size)
        self.init_c = nn.Linear(encoder_dim, hidden_size)

    def forward(self, encoder_out, captions,states=None):
        batch_size = encoder_out.size(0)
        seq_len = captions.size(1)
        vocab_size = self.linear.out_features

        # Init hidden state and cell state
        h = self.init_h(encoder_out.mean(dim=1))  # (B, hidden_size)
        c = self.init_c(encoder_out.mean(dim=1))

        outputs = torch.zeros(batch_size, seq_len - 1, vocab_size).to(encoder_out.device)

        for t in range(seq_len - 1):
            embeddings = self.dropout(self.embed(captions[:, t]))  # (B, embed_size)
            context, _ = self.attention(encoder_out, h)            # (B, encoder_dim)
            lstm_input = torch.cat([embeddings, context], dim=1)   # (B, embed + encoder_dim)
            h, c = self.lstm(lstm_input, (h, c))                   # LSTMCell
            output = self.linear(self.dropout(h))                 # (B, vocab_size)
            outputs[:, t, :] = output

        return outputs,states

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize models
encoder = EncoderDeiT().to(device)
decoder = DecoderRNN(embed_size=256, hidden_size=512, vocab_size=len(word2idx),attention_dim=256, encoder_dim=384, num_layers=1,dropout_prob=0.3).to(device)

# Load model weights
encoder.load_state_dict(torch.load('encoder_finalv1_att_final_deit.pth'))
decoder.load_state_dict(torch.load('decoder_finalv1_att_final_deit.pth'))
encoder.eval()
decoder.eval()

# Beam Search Decoder
def beam_search_decoder(predictions, k):
    sequences = [[list(), 1.0]]  # list of (sequence, score)
    for row in predictions:
        all_candidates = list()
        for i in range(len(sequences)):
            seq, score = sequences[i]
            for j in range(len(row)):
                candidate = [seq + [j], score * -np.log(row[j])]
                all_candidates.append(candidate)
        # order all candidates by score
        ordered = sorted(all_candidates, key=lambda tup: tup[1])
        # select k best
        sequences = ordered[:k] 
    return sequences

# Function to generate a caption
def generate_caption(image_path, encoder, decoder, idx2word, word2idx, transform, beam_width=3, max_len=40):
    image = Image.open(image_path).convert("RGB")
    if transform:
        image = transform(image)
    image = image.unsqueeze(0).to(device)

    encoder.eval()
    decoder.eval()

    with torch.no_grad():
        encoder_out = encoder(image)  # (1, 196, 2048)
        encoder_mean = encoder_out.mean(dim=1)
        h = decoder.init_h(encoder_mean)  # (1, hidden_size)
        c = decoder.init_c(encoder_mean)

        # Beam starts with <START> token
        sequences = [([word2idx['<START>']], 0.0, h, c)]

        for _ in range(max_len):
            all_candidates = []
            for seq, score, h_t, c_t in sequences:
                last_token = seq[-1]
                if last_token == word2idx['<END>']:
                    all_candidates.append((seq, score, h_t, c_t))
                    continue

                # Embed last token only
                last_token_tensor = torch.tensor([last_token]).to(device)
                embedding = decoder.embed(last_token_tensor)  # (1, embed_size)
                context, _ = decoder.attention(encoder_out, h_t)
                lstm_input = torch.cat([embedding, context], dim=1)
                h_new, c_new = decoder.lstm(lstm_input, (h_t, c_t))
                output = decoder.linear(h_new)
                probs = torch.softmax(output, dim=1)

                top_k_probs, top_k_idxs = probs.topk(beam_width)

                for i in range(beam_width):
                    idx = top_k_idxs[0, i].item()
                    prob = top_k_probs[0, i].item()
                    new_seq = seq + [idx]
                    new_score = score + np.log(prob + 1e-12)  # log-prob (avoid log(0))
                    all_candidates.append((new_seq, new_score, h_new.clone(), c_new.clone()))

            sequences = sorted(all_candidates, key=lambda tup: tup[1], reverse=True)[:beam_width]

            # Optional: break early if all sequences end with <END>
            if all(seq[-1] == word2idx['<END>'] for seq, _, _, _ in sequences):
                break

        # Best sequence
        best_seq = sequences[0][0]
        caption = ''.join([idx2word[str(idx)] for idx in best_seq if idx not in [word2idx['<START>'], word2idx['<END>'], word2idx['<PAD>']]])
        return caption


# Transformations
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

# Image path
image_path = '/home/vitoupro/code/image_captioning/data/8.png'

caption = generate_caption(image_path, encoder, decoder, idx2word, word2idx, transform)
print("Generated Caption:", caption.replace(" ", "")) 


Generated Caption: បក្សីពណ៌សទំលើមែកផ្កា
